# Notebook 06 — Evaluation & Interpretation
Load the saved best models from notebooks 04 and 05.
Produce: confusion matrices, classification reports, feature importances, SHAP values, and final comparison table.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import sys
sys.path.append('../')
from utils import classification_metrics
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
sns.set_theme(style='whitegrid')

PROCESSED = '../data/processed/'

with open(PROCESSED + 'best_disaster_model.pkl', 'rb') as f:
    d_bundle = pickle.load(f)

with open(PROCESSED + 'best_project_model.pkl', 'rb') as f:
    p_bundle = pickle.load(f)

print('Disaster model:', d_bundle['model_name'])
print('Project  model:', p_bundle['model_name'])

## 6.1 Confusion Matrices — Both Models
Rows = actual class, Columns = predicted class.
Off-diagonal cells are misclassifications — check whether errors are adjacent tiers (acceptable) or extreme jumps (problematic).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

for ax, bundle, cmap, title in [
    (axes[0], d_bundle, 'Blues',   'Disaster-Level Model'),
    (axes[1], p_bundle, 'Oranges', 'Project-Level Model'),
]:
    cm   = confusion_matrix(bundle['y_test'], bundle['preds'])
    disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                                  display_labels=bundle['target_names'])
    disp.plot(ax=ax, colorbar=False, cmap=cmap)
    ax.set_title(f'{bundle["model_name"]} — {title}', fontsize=11)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=20, ha='right', fontsize=8)
    ax.set_yticklabels(ax.get_yticklabels(), fontsize=8)

plt.tight_layout()
plt.savefig(PROCESSED + 'confusion_matrices.png', dpi=150)
plt.show()

## 6.2 Classification Reports — Per-Class Precision, Recall, F1

In [ ]:
for bundle in [d_bundle, p_bundle]:
    print(f"\n{'='*55}")
    print(f"  {bundle['model_name']} — {bundle['level'].title()}-Level")
    print(f"{'='*55}")
    print(classification_report(bundle['y_test'], bundle['preds'],
                                target_names=bundle['target_names'],
                                zero_division=0))

## 6.3 Final Metrics Comparison Table

In [ ]:
rows = []
for bundle in [d_bundle, p_bundle]:
    m = classification_metrics(bundle['y_test'].values, bundle['preds'],
                               label=f"{bundle['model_name']} ({bundle['level'].title()}-Level)",
                               target_names=bundle['target_names'])
    m['Level'] = bundle['level'].title()
    rows.append(m)

comparison = pd.DataFrame(rows)[['label', 'Level', 'Accuracy', 'F1_weighted']]
comparison = comparison.rename(columns={'label': 'Model'})
comparison

## 6.4 SHAP Values — Why Does the Model Predict What It Predicts?
SHAP explains individual predictions — which features push a value up or down.
Install with: `pip install shap`


In [ ]:
try:
    import shap

    bundle   = d_bundle
    pipeline = bundle['pipeline']
    X_test   = bundle['X_test']

    X_transformed = pipeline.named_steps['pre'].transform(X_test)

    pre       = pipeline.named_steps['pre']
    ohe_names = pre.transformers_[0][1].named_steps['ohe'].get_feature_names_out(bundle['cat_features'])
    all_names = list(ohe_names) + bundle['num_features']

    rf_model  = pipeline.named_steps['model']
    explainer = shap.TreeExplainer(rf_model)

    idx = np.random.choice(len(X_transformed), size=min(300, len(X_transformed)), replace=False)
    shap_values = explainer.shap_values(X_transformed[idx])

    # For multiclass RF, shap_values is a list — show the most populated class
    if isinstance(shap_values, list):
        shap_values = shap_values[1]   # Class 1 = Moderate (most populated disaster tier)

    shap.summary_plot(shap_values, X_transformed[idx],
                      feature_names=all_names, max_display=15, show=True)

except ImportError:
    print('shap not installed. Run: pip install shap')

## 6.5 Critical Reflection

### Problem Framing
We reframed this as a **multi-class classification problem** using 4 funding tiers anchored
to real US federal policy thresholds (FEMA PA small/large project threshold + FAR tiers).
This is more actionable than regression: stakeholders need to know *which budget category*
a disaster falls into, not a precise dollar figure that carries false precision.

### What the models do well
- The disaster-level model captures broad patterns: incident type, affected counties,
  and state demographics explain a meaningful portion of tier variance.
- Classification is more robust to extreme outliers (Katrina, Harvey) than regression —
  those events land in Tier 3 regardless of exact dollar amount.

### Class Imbalance
- Most projects are Small/Micro; Major projects are rare.
- We used `class_weight='balanced'` to prevent models from ignoring minority classes.
- The confusion matrix reveals whether rare high-tier events are being systematically missed.

### Limitations
- **Adjacent tier confusion**: Some predictions land one tier off — this is acceptable
  (predicting Moderate when true is Major) vs. extreme misses (Micro vs. Major).
- **Policy drift**: FEMA's cost-share rules changed over the dataset's 25-year span.
  The $131,100 threshold is the 2019 value; earlier years used lower thresholds.
- **Geographic mismatch**: Projects spanning multiple counties are assigned one FIPS.

### Future Work
- **XGBoost/LightGBM tuning** for higher F1 on minority classes.
- **NOAA Storm Events** (wind speed, precipitation) to improve physical intensity signal.
- **Real-time API pipeline**: predict tier for newly declared disasters via FEMA OpenFEMA API.
- **Ordinal classification**: exploit the natural order of tiers (0 < 1 < 2 < 3) using
  ordinal loss functions rather than treating classes as independent.